In [1]:
import pandas as pd
import requests
import json
import re
from tqdm import tqdm

# Membaca dataset
try:
    df_train = pd.read_csv('../dataset/train.csv')
    print("Dataset berhasil dimuat!")
    print(f"Total data: {df_train.shape[0]} baris")
    display(df_train.head(3)) # Menampilkan 3 baris pertama
except FileNotFoundError:
    print("Error: File train.csv tidak ditemukan. Pastikan lokasinya sudah benar.")

Dataset berhasil dimuat!
Total data: 17307 baris


,essay_id,full_text,score
0,000d118,Many people have car where they live. The thin...,3
1,000fe60,I am a scientist at NASA that is discussing th...,3
2,001ab80,People always wish they had the same technolog...,4


In [2]:
def test_koneksi_ollama():
    url = "http://localhost:11434/api/generate"
    
    # Payload adalah paket pesan yang kita kirim ke AI
    payload = {
        "model": "llama3", 
        "prompt": "Halo! Jawab dengan satu kalimat pendek: apakah kamu sudah siap menilai esai?",
        "stream": False
    }
    
    print("Mengirim pesan ke Ollama...")
    try:
        response = requests.post(url, json=payload)
        hasil = response.json().get('response', '')
        print("\nJawaban Ollama:")
        print(f"🤖 {hasil}")
    except Exception as e:
        print(f"\nGagal terhubung ke Ollama. Error: {e}")

test_koneksi_ollama()

Mengirim pesan ke Ollama...

Jawaban Ollama:
🤖 Siap!


## Zero-Shot

In [3]:
import re

def get_llm_score(prompt):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": "llama3",
        "prompt": prompt,
        "stream": False,
        "temperature": 0.0 # Angka 0 agar jawaban LLM konsisten dan tidak berhalusinasi
    }
    
    try:
        response = requests.post(url, json=payload)
        hasil_teks = response.json().get('response', '')
        
        # Ekstraksi menggunakan Regex: Cari angka 1 sampai 6 di dalam jawaban AI
        match = re.search(r'[1-6]', hasil_teks)
        if match:
            return int(match.group())
        else:
            return None # Jika AI bandel dan tidak mengeluarkan angka
    except Exception as e:
        print(f"Error LLM: {e}")
        return None

In [4]:
# 1. Ambil 20 data pertama untuk eksperimen cepat
df_eksperimen = df_train.head(20).copy()
prediksi_zero = []

print("Memulai penilaian Zero-Shot pada 20 esai...")

# 2. Looping (tqdm akan memunculkan progress bar yang keren)
for index, row in tqdm(df_eksperimen.iterrows(), total=df_eksperimen.shape[0]):
    esai = row['full_text']
    
    # 3. Rancang System Prompt yang sangat ketat
    prompt_zero = f"""Anda adalah sistem penilai esai otomatis yang sangat objektif.
Tugas Anda adalah memberikan skor pada esai berikut dari skala 1 hingga 6.
(1 = sangat buruk, 6 = sangat sempurna).
    
ATURAN WAJIB: HANYA berikan SATU ANGKA (1, 2, 3, 4, 5, atau 6). Jangan berikan teks awalan, jangan berikan penjelasan, jangan gunakan tanda baca.
    
Esai:
{esai}
    
Skor:"""
    
    # 4. Panggil LLM
    skor = get_llm_score(prompt_zero)
    
    # 5. Penanganan error (Fallback)
    if skor is None:
        skor = 3 # Nilai tengah jika LLM gagal menjawab sesuai format
        
    prediksi_zero.append(skor)

# 6. Simpan hasil ke dalam kolom baru
df_eksperimen['zero_shot_score'] = prediksi_zero
print("\nPenilaian Zero-Shot selesai!")

# Tampilkan perbandingan nilai asli guru vs tebakan AI
display(df_eksperimen[['essay_id', 'score', 'zero_shot_score']])

Memulai penilaian Zero-Shot pada 20 esai...


100%|██████████| 20/20 [00:19<00:00,  1.02it/s]


Penilaian Zero-Shot selesai!


,essay_id,score,zero_shot_score
0,000d118,3,4
1,000fe60,3,4
2,001ab80,4,4
3,001bdc0,4,4
4,002ba53,3,4
5,0030e86,4,5
6,0033037,2,4
7,0033bf4,3,4
8,0036253,2,4
9,0040e27,3,4


In [5]:
from sklearn.metrics import cohen_kappa_score

# Hitung QWK antara nilai asli ('score') dan nilai prediksi AI ('zero_shot_score')
qwk_zero = cohen_kappa_score(
    df_eksperimen['score'], 
    df_eksperimen['zero_shot_score'], 
    weights='quadratic'
)

print(f"Skor QWK Zero-Shot (Sampel 20 data): {qwk_zero:.4f}")
print("(Skor sempurna adalah 1.0. Jika skor masih di bawah 0.5, itu wajar untuk Zero-Shot)")

Skor QWK Zero-Shot (Sampel 20 data): 0.0887
(Skor sempurna adalah 1.0. Jika skor masih di bawah 0.5, itu wajar untuk Zero-Shot)


## Few-Shot

In [6]:
# Fungsi untuk memotong teks agar tidak terlalu panjang di dalam prompt
def potong_teks(teks, batas_kata=50):
    kata = teks.split()
    if len(kata) > batas_kata:
        return " ".join(kata[:batas_kata]) + "..."
    return teks

# Mengambil otomatis 1 contoh skor 2 dan 1 contoh skor 5 dari df_train
contoh_esai_2 = df_train[df_train['score'] == 2].iloc[0]['full_text']
contoh_esai_5 = df_train[df_train['score'] == 5].iloc[0]['full_text']

# Memotong contoh agar lebih ringkas
contoh_2_pendek = potong_teks(contoh_esai_2)
contoh_5_pendek = potong_teks(contoh_esai_5)

print("Contoh esai untuk Few-Shot berhasil disiapkan!")

Contoh esai untuk Few-Shot berhasil disiapkan!


In [7]:
prediksi_few = []

print("Memulai penilaian Few-Shot pada 20 esai...")

for index, row in tqdm(df_eksperimen.iterrows(), total=df_eksperimen.shape[0]):
    esai_target = row['full_text']
    
    # Rancang Prompt Few-Shot dengan menyisipkan contoh
    prompt_few = f"""Anda adalah penilai esai otomatis. Berikan skor dari skala 1 hingga 6.
ATURAN WAJIB: HANYA berikan SATU ANGKA (1, 2, 3, 4, 5, atau 6). Jangan berikan teks tambahan apapun.

Berikut adalah referensi standar penilaian:

[Contoh Esai Buruk]
Esai: {contoh_2_pendek}
Skor: 2

[Contoh Esai Sangat Baik]
Esai: {contoh_5_pendek}
Skor: 5

Sekarang, nilai esai di bawah ini:
Esai: {esai_target}
Skor:"""
    
    # Panggil fungsi LLM yang sudah kita buat sebelumnya
    skor = get_llm_score(prompt_few)
    
    if skor is None:
        skor = 3 # Fallback
        
    prediksi_few.append(skor)

# Simpan hasil prediksi Few-Shot
df_eksperimen['few_shot_score'] = prediksi_few
print("\nPenilaian Few-Shot selesai!")

# Tampilkan perbandingan lengkap
display(df_eksperimen[['essay_id', 'score', 'zero_shot_score', 'few_shot_score']])

Memulai penilaian Few-Shot pada 20 esai...


100%|██████████| 20/20 [00:18<00:00,  1.05it/s]


Penilaian Few-Shot selesai!


,essay_id,score,zero_shot_score,few_shot_score
0,000d118,3,4,4
1,000fe60,3,4,4
2,001ab80,4,4,4
3,001bdc0,4,4,5
4,002ba53,3,4,5
5,0030e86,4,5,5
6,0033037,2,4,3
7,0033bf4,3,4,4
8,0036253,2,4,4
9,0040e27,3,4,4


In [8]:
# Hitung QWK untuk Few-Shot
qwk_few = cohen_kappa_score(
    df_eksperimen['score'], 
    df_eksperimen['few_shot_score'], 
    weights='quadratic'
)

print("=== HASIL EVALUASI QWK (20 Sampel) ===")
print(f"Skor Zero-Shot : {qwk_zero:.4f}")
print(f"Skor Few-Shot  : {qwk_few:.4f}")

# Cek peningkatan
selisih = qwk_few - qwk_zero
if selisih > 0:
    print(f"\nKesimpulan: Metode Few-Shot LEBIH BAIK dengan peningkatan skor sebesar {selisih:.4f}")
elif selisih < 0:
    print(f"\nKesimpulan: Metode Zero-Shot lebih baik (perlu optimasi prompt contoh).")
else:
    print("\nKesimpulan: Kedua metode memberikan hasil yang sama.")

=== HASIL EVALUASI QWK (20 Sampel) ===
Skor Zero-Shot : 0.0887
Skor Few-Shot  : 0.2892

Kesimpulan: Metode Few-Shot LEBIH BAIK dengan peningkatan skor sebesar 0.2005


## File Submission

In [9]:
# 1. Baca data uji (test.csv)
try:
    df_test = pd.read_csv('../dataset/test.csv')
    print(f"Data test berhasil dimuat: {df_test.shape[0]} esai siap dinilai.")
except FileNotFoundError:
    print("Error: File test.csv tidak ditemukan.")

prediksi_final = []

print("Memulai penilaian untuk Submission Kaggle...")

# 2. Looping menggunakan Few-Shot (karena terbukti lebih akurat)
for index, row in tqdm(df_test.iterrows(), total=df_test.shape[0]):
    esai_target = row['full_text']
    
    # Gunakan prompt Few-Shot yang persis sama seperti tahap sebelumnya
    prompt_final = f"""Anda adalah penilai esai otomatis. Berikan skor dari skala 1 hingga 6.
ATURAN WAJIB: HANYA berikan SATU ANGKA (1, 2, 3, 4, 5, atau 6). Jangan berikan teks tambahan apapun.

[Contoh Esai Buruk]
Esai: {contoh_2_pendek}
Skor: 2

[Contoh Esai Sangat Baik]
Esai: {contoh_5_pendek}
Skor: 5

Sekarang, nilai esai di bawah ini:
Esai: {esai_target}
Skor:"""
    
    skor = get_llm_score(prompt_final)
    
    if skor is None:
        skor = 3 # Fallback
        
    prediksi_final.append(skor)

# 3. Format hasil sesuai aturan Kaggle (Hanya butuh 2 kolom: essay_id dan score)
submission_df = pd.DataFrame({
    'essay_id': df_test['essay_id'],
    'score': prediksi_final
})

# 4. Simpan menjadi file CSV
submission_df.to_csv('../outputs/submission.csv', index=False)
print("\nSukses! File 'submission.csv' berhasil dibuat dan siap diunggah ke Kaggle.")

# Tampilkan cuplikan hasilnya
display(submission_df.head())

Data test berhasil dimuat: 3 esai siap dinilai.
Memulai penilaian untuk Submission Kaggle...


100%|██████████| 3/3 [00:03<00:00,  1.15s/it]


Sukses! File 'submission.csv' berhasil dibuat dan siap diunggah ke Kaggle.


,essay_id,score
0,000d118,4
1,000fe60,3
2,001ab80,4
